# Consolidated Customer Segmentation and Proposed Perks

This notebook combines the recommended four-group full-cohort clustering with the existing provisional perk policy. It produces one customer-level delivery table and audience-facing profiles. It does not rerun K-means, alter source data, or validate favorite perks.

**Prerequisites:** outputs from notebooks 06 and 07, plus the study feature table.

**Outputs:** `reports/consolidated_segmentation/`. Customer segment labels reflect relative observed activity. Perk recommendations remain clearly marked as hypotheses. The analytical model is engagement/frequency clustering, not classic RFM/LRFM; earlier models are retained as comparisons.

Repeated runs replace only this notebook's own derived outputs. Dependencies: pandas.


In [ ]:
from pathlib import Path
import pandas as pd

print("STEP 1: Load study features, full-cohort clusters and proposed perks", flush=True)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "own_processed" / "user_features_windowed.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Expected data/own_processed/user_features_windowed.csv. No output written.")
SOURCE = PROJECT / "data" / "own_processed" / "user_features_windowed.csv"
CLUSTERS = PROJECT / "reports" / "full_cohort_clustering" / "full_cohort_candidate_assignments.csv"
PERKS = PROJECT / "reports" / "provisional_perk_assignments" / "customer_perk_assignments_provisional.csv"
OUT = PROJECT / "reports" / "consolidated_segmentation"
for path in [SOURCE, CLUSTERS, PERKS]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}. No output written.")
s = pd.read_csv(SOURCE, dtype={"user_id":"string"})
c = pd.read_csv(CLUSTERS, usecols=["user_id", "activity_k4"], dtype={"user_id":"string"})
p = pd.read_csv(PERKS, dtype={"user_id":"string"})
for frame in [s,c,p]:
    assert frame.user_id.is_unique
assert set(s.user_id)==set(c.user_id)==set(p.user_id)
keep = ["user_id", "proposed_perk", "assignment_reason", "evidence_confidence",
        "assignment_is_fallback", "eligible_rule_count", "eligible_perks",
        "priority_resolved_overlap", "preference_verified"]
d = s.merge(c,on="user_id",validate="one_to_one").merge(p[keep],on="user_id",validate="one_to_one")
assert d.activity_k4.notna().all() and d.proposed_perk.notna().all()
assert d.activity_k4.nunique()==4

print("STEP 2: Label the four clusters from their observed profiles", flush=True)
medians = d.groupby("activity_k4")[["session_recency_days","rfm_frequency_trips"]].median()
ordered = medians.sort_values(["rfm_frequency_trips", "session_recency_days"], ascending=[False,True]).index.tolist()
high, low = ordered[:2], ordered[2:]
assert medians.loc[high,"rfm_frequency_trips"].min() > medians.loc[low,"rfm_frequency_trips"].max(), "Median booking activity does not support the proposed labels."
high = sorted(high,key=lambda k:medians.loc[k,"session_recency_days"])
low = sorted(low,key=lambda k:medians.loc[k,"session_recency_days"])
labels = {high[0]:("S1","Higher booking activity / more recent engagement"),
          high[1]:("S2","Higher booking activity / less recent engagement"),
          low[0]:("S3","Lower booking activity / more recent engagement"),
          low[1]:("S4","Lower booking activity / less recent engagement")}
d["segment_id"] = d.activity_k4.map(lambda k:labels[k][0])
d["segment_name"] = d.activity_k4.map(lambda k:labels[k][1])
d["segment_method"] = "K-means: engagement recency and recent noncanceled-trip frequency"
d["segmentation_status"] = "Recommended four-cluster solution; analytical approval pending"
d["perk_status"] = "Provisional pilot policy; favorite perk unverified"
rows = []
for segment,sub in d.groupby("segment_id"):
    known = sub.rfm_monetary_estimate.dropna()
    rows.append({"segment_id":segment,"segment_name":sub.segment_name.iloc[0],
                 "customers":len(sub),"share_of_full_cohort":len(sub)/len(d),
                 "median_session_recency_days":sub.session_recency_days.median(),
                 "median_recent_noncanceled_trips":sub.rfm_frequency_trips.median(),
                 "customers_with_known_recent_booking_value":len(known),
                 "recent_booking_value_coverage":len(known)/len(sub),
                 "median_known_recent_booking_value_usd":known.median(),
                 "customers_with_no_observed_bookings":int(sub.booking_status.eq("No observed bookings").sum()),
                 "customers_with_only_canceled_bookings":int(sub.booking_status.eq("Only canceled bookings").sum()),
                 "customers_with_unknown_recent_booking_value":int(sub.rfm_monetary_estimate.isna().sum()),
                 "generic_fallback_perk_customers":int(sub.assignment_is_fallback.sum())})
profiles = pd.DataFrame(rows)
perk_mix = d.groupby(["segment_id","segment_name","proposed_perk"]).size().reset_index(name="customers")
perk_mix["share_of_segment"] = perk_mix.customers / perk_mix.segment_id.map(profiles.set_index("segment_id").customers)
assert profiles.customers.sum()==len(s)==perk_mix.customers.sum()
assert not d.preference_verified.any()

print("STEP 3: Save one customer-level delivery table and audience-facing notes", flush=True)
OUT.mkdir(parents=True,exist_ok=True)
export_cols = ["user_id","segment_id","segment_name","segment_method","segmentation_status",
    "proposed_perk","perk_status","assignment_reason","evidence_confidence",
    "assignment_is_fallback","eligible_rule_count","eligible_perks","priority_resolved_overlap",
    "preference_verified","booking_status","baseline_status","session_recency_days",
    "rfm_frequency_trips","rfm_monetary_estimate","rfm_recency_days","lrfm_length_days",
    "recent_spend_complete","recent_potential_exposure_days","analysis_reference_date",
    "feature_window_start","feature_window_end"]
d[export_cols].sort_values("user_id").to_csv(OUT / "customer_segments_and_perks_draft.csv",index=False)
profiles.to_csv(OUT / "segment_profiles.csv",index=False)
perk_mix.to_csv(OUT / "segment_perk_mix.csv",index=False)
lines = ["# Recommended Segmentation and Provisional Perk Policy", "",
         f"All {len(d):,} study customers have a K-means segment and a proposed invitation benefit. Four analytical clusters are recommended; perk assignments remain provisional.",
         "", "## Recommended analytical design", "Use full-cohort engagement-recency and recent noncanceled-trip-frequency clustering as the primary segmentation. Retain RFM/LRFM models as benchmarks and profile booking expenditure, booking recency and observed relationship span wherever defined. The main model is not classic RFM or LRFM. The choice prioritizes full-cohort coverage and interpretable activity patterns rather than claiming superiority from silhouette values in different feature spaces.",
         "", "## Segment profiles", "```", profiles[["segment_id","customers","median_session_recency_days","median_recent_noncanceled_trips","recent_booking_value_coverage"]].round(3).to_string(index=False), "```"]
lines += [f"- {r.segment_id}: {r.segment_name}." for r in profiles.itertuples()]
lines += ["", "## Meaning of the labels", "Higher/lower and more/less recent are relative comparisons within the selected cohort, not universal customer tiers. Cluster medians do not describe every member. Less recent engagement does not establish churn or dormancy. No business, leisure, family or luxury identities are inferred.",
          "", "## Proposed perk recommendations", "Proposed perks are assigned individually by the documented notebook 06 pilot rules. They are not clustering inputs or verified favorites; a cluster can contain multiple proposed perks. Eligibility overlap and fallback labels are retained. Unknown spending does not force a special customer segment. Customers without noncanceled bookings belong to lower-booking-activity clusters while keeping explicit booking statuses.",
          "", "## Validation and limitations", "Within the all-customer candidate family, four groups had the highest sample silhouette (approximately 0.398), with minimum seed-agreement ARI approximately 0.976 across tested seeds. These are internal, exploratory checks, not evidence of future stability or retention gains. Sessions define engagement recency, not booking recency. Recent booking frequency covers January 5–July 28, 2023; tenure exposure remains unequal. Spending coverage is reported and unknown values are not replaced by zero. The selected cohort is already highly website-engaged by its eligibility filter and is not representative of all TravelTide customers. The current perk policy is sensitive: stricter thresholds changed 33.5% of recommendations in the study comparison; placing the combined flight–hotel rule before baggage changed 39.9% in an additional priority check on the uploaded study table. These results argue for treating the policy as a pilot, not an optimized favorite-perk model.",
          "", "## Presentation recommendation", "Mention all four analytical segments. For approximately three detailed examples, consider S1 (recent engagement with higher booking activity), S3 (recent engagement with lower booking activity), and S2 (less recent engagement with higher booking activity). Justify invitation timing and proposed benefits using individual behavior. S4 remains visible in the overview and customer export. These examples are recommendations, not mandatory highlights.",
          "", "## Suggested approach statement", "The project compares RFM and LRFM and recommends a full-cohort activity segmentation using K-means on engagement recency and recent booking frequency. Booking expenditure, relationship length and travel behavior support profiling and provisional perk recommendations. Recommendations are hypotheses for a rewards-program invitation pilot, not proven individual preferences or causal retention effects.",
          "", "## Deliverables", "customer_segments_and_perks_draft.csv contains one row per customer. segment_profiles.csv records group counts and measurement coverage. segment_perk_mix.csv summarizes the proposed benefits within each analytical cluster. All tables retain the distinction between segmentation and a pilot recommendation policy."]
(OUT / "segmentation_delivery_summary.md").write_text("\n".join(lines),encoding="utf-8")
print("COMPLETE:",OUT,flush=True)
print("All source files and prior model outputs were preserved.",flush=True)
